<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/rl/01_bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R1 探索と活用:バンディット問題(notebook)

本文:[`01-bandit.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/01-bandit.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

class GaussianBandit:
    """K 台のスロットマシン。台 a の報酬は、平均 mu[a]、標準偏差 sigma[a] の正規分布から出る。"""
    def __init__(self, seed=None, K=5):
        param_rng = np.random.default_rng(seed)
        self.K = K
        self.mu = param_rng.uniform(0.0, 10.0, size=K)      # エージェントは知らない
        self.sigma = param_rng.uniform(0.5, 2.0, size=K)
        self.reset(seed)

    def reset(self, seed=None):
        self.rng = np.random.default_rng(seed)               # 報酬の乱数を初めに戻す

    def step(self, a):
        return float(self.rng.normal(self.mu[a], self.sigma[a]))

    @property
    def optimal_action(self):
        return int(np.argmax(self.mu))

## 0. 環境:5台のスロットマシン

`seed=2026` の 2026 を、自分の学籍番号の数字に変えてから実行してよい(人によって違う問題になる)。

In [ ]:
env = GaussianBandit(seed=2026)
for a in range(env.K):
    print(f"台 {a}: 平均 {env.mu[a]:5.2f}  標準偏差 {env.sigma[a]:4.2f}")
print("最適な台:", env.optimal_action)

xs = np.linspace(env.mu.min() - 5, env.mu.max() + 5, 400)
for a in range(env.K):
    plt.plot(xs, np.exp(-(xs - env.mu[a])**2 / (2 * env.sigma[a]**2)) / (np.sqrt(2 * np.pi) * env.sigma[a]), label=f"a{a}")
plt.xlabel("reward"); plt.ylabel("density"); plt.legend(); plt.title("reward distributions"); plt.show()

## 1. 標本平均を逐次的に計算する

本文の式 (1): $Q_n = Q_{n-1} + \frac{1}{n}(r_n - Q_{n-1})$ 。

**手計算**:本文の確認課題1(報酬 3, 5, 4, 8)で $Q_1, \dots, Q_4$ を計算する。

- 手計算:(ここに書く)

**実装**:1つ前の推定 `Q`、回数 `n`(今回を含む)、今回の報酬 `r` から、新しい推定を返す関数を書く。

In [ ]:
def update_mean(Q, n, r):
    raise NotImplementedError("TODO: Q + (r - Q) / n を返す")

In [ ]:
Q = 0.0
history = []
for n, r in enumerate([3, 5, 4, 8], start=1):
    Q = update_mean(Q, n, r)
    history.append(Q)
print("Q_1..Q_4 =", history)
check("Q_1..Q_4", history, [3, 4, 4, 5])
rewards = np.random.default_rng(0).normal(5, 2, 1000)
Q = 0.0
for n, r in enumerate(rewards, start=1):
    Q = update_mean(Q, n, r)
check("1000個の報酬でも、最後の値は平均と一致する", Q, rewards.mean(), tol=1e-9)

## 2. ε-greedy の確率

本文の式 (2) を実装する。同点の行動が複数あるときは、その中で $(1 - \varepsilon)$ を等分する。

**手計算**:本文の確認課題2($\varepsilon = 0.1$、 $Q = (2, 5, 5, 1, 0)$)で、各行動の確率を求める。

- 手計算:(ここに書く)

In [ ]:
def epsilon_greedy_probs(Q, epsilon):
    K = len(Q)
    probs = np.full(K, epsilon / K)
    raise NotImplementedError("TODO: Q が最大の行動(np.flatnonzero(Q == Q.max()))に (1 - epsilon) / その個数 を足す")
    return probs

In [ ]:
check("同点が2つ", epsilon_greedy_probs(np.array([2., 5., 5., 1., 0.]), 0.1), [0.02, 0.47, 0.47, 0.02, 0.02])
check("最初はすべて同点なので一様", epsilon_greedy_probs(np.zeros(5), 0.1), np.full(5, 0.2))
check("epsilon = 0 なら最大の行動だけ", epsilon_greedy_probs(np.array([1., 3., 2.]), 0.0), [0, 1, 0])
check("確率の和は 1", epsilon_greedy_probs(np.random.default_rng(0).normal(size=7), 0.3).sum(), 1.0)

## 3. ε-greedy で1回実行する

**予想**:$\varepsilon = 0$(greedy)と $\varepsilon = 0.1$ で、1000 回引いたとき、最適な台を選ぶ割合はどちらが大きいか。

- 予想:(ここに書く)

**実装**:下の関数の `TODO` の1行(価値の更新)を書く。`alpha=None` なら標本平均(ステップサイズ $1/N$)、数を与えたら一定のステップサイズを使う。

In [ ]:
def run_eps_greedy(env, epsilon, T=1000, Q0=0.0, alpha=None, seed=0):
    rng = np.random.default_rng(seed)
    env.reset(seed=seed)
    Q = np.full(env.K, float(Q0))
    N = np.zeros(env.K)
    rewards, actions = np.zeros(T), np.zeros(T, dtype=int)
    Q_hist = np.zeros((T + 1, env.K)); Q_hist[0] = Q
    for t in range(T):
        a = int(rng.choice(env.K, p=epsilon_greedy_probs(Q, epsilon)))
        r = env.step(a)
        N[a] += 1
        step_size = 1.0 / N[a] if alpha is None else alpha
        raise NotImplementedError("TODO: Q[a] を step_size を使って更新する(本文の式 (1) の 1/n を step_size にしたもの)")
        rewards[t], actions[t] = r, a
        Q_hist[t + 1] = Q
    return dict(Q=Q, N=N, rewards=rewards, actions=actions, Q_hist=Q_hist)

In [ ]:
res = run_eps_greedy(env, epsilon=0.1, seed=0)
check("選んだ回数の合計は 1000", res["N"].sum(), 1000)
played = res["N"] > 0
mean_by_arm = np.array([res["rewards"][res["actions"] == a].mean() for a in np.flatnonzero(played)])
check("ステップサイズ 1/N なら、Q は台ごとの報酬の平均", res["Q"][played], mean_by_arm, tol=1e-9)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a in range(env.K):
    ax[0].plot(res["Q_hist"][:, a], label=f"a{a} (mu={env.mu[a]:.1f})")
ax[0].set_xlabel("step"); ax[0].set_ylabel("Q(a)"); ax[0].legend(fontsize=8)
ax[1].bar(range(env.K), res["N"]); ax[1].set_xlabel("action"); ax[1].set_ylabel("N(a)")
plt.tight_layout(); plt.show()
for eps in [0.0, 0.1]:
    r = run_eps_greedy(env, epsilon=eps, seed=0)
    print(f"epsilon = {eps}: 最適な台を選んだ割合 {np.mean(r['actions'] == env.optimal_action):.3f}  平均の報酬 {r['rewards'].mean():.3f}")

seed を変えて、上のセルを何度か実行してみる。greedy($\varepsilon = 0$)は、seed によって結果が大きく変わることがある。たまたま最初に良い報酬が出た台にこだわるからである。

## 4. UCB

本文の式 (3):$Q_t(a) + c\sqrt{\ln t / N_t(a)}$ を計算する関数を書く。$N_t(a) = 0$ の行動は先に試すので、この関数には $N_t(a) \ge 1$ の場合だけが渡される。

In [ ]:
def ucb_scores(Q, N, t, c):
    raise NotImplementedError("TODO: Q + c * sqrt(log(t) / N) を配列のまま計算する")

In [ ]:
check("UCB のスコア", ucb_scores(np.array([1.0, 2.0]), np.array([1.0, 4.0]), np.e**4, 1.0), [3.0, 3.0])

def run_ucb(env, c=2.0, T=1000, seed=0):
    rng = np.random.default_rng(seed)
    env.reset(seed=seed)
    Q, N = np.zeros(env.K), np.zeros(env.K)
    rewards, actions = np.zeros(T), np.zeros(T, dtype=int)
    for t in range(T):
        untried = np.flatnonzero(N == 0)
        if len(untried) > 0:
            a = int(rng.choice(untried))                       # まだ試していない行動を先に
        else:
            s = ucb_scores(Q, N, t + 1, c)
            a = int(rng.choice(np.flatnonzero(s == s.max())))  # 同点はランダムに
        r = env.step(a)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards[t], actions[t] = r, a
    return dict(Q=Q, N=N, rewards=rewards, actions=actions)

res = run_ucb(env, c=2.0, seed=0)
print("UCB で各台を選んだ回数:", res["N"].astype(int), " 最適な台:", env.optimal_action)

## 5. 複数回の実行の平均で比べる

1回の実行の結果は、乱数で大きく変わる。seed を変えて 100 回実行し、各時刻の(1)平均の報酬、(2)最適な台を選んだ割合を比べる(本文 §4)。少し時間がかかる(数十秒)。

**予想**:4つの方法のうち、1000 ステップ目で最適な台を選ぶ割合がいちばん高いのはどれか。いちばん低いのはどれか。

- 予想:(ここに書く)

In [ ]:
def average_runs(run, n_runs=100, **kw):
    rewards, optimal = 0, 0
    for k in range(n_runs):
        r = run(env, seed=k, **kw)
        rewards = rewards + r["rewards"]
        optimal = optimal + (r["actions"] == env.optimal_action)
    return rewards / n_runs, optimal / n_runs

configs = {
    "greedy, Q0=0": (run_eps_greedy, dict(epsilon=0.0)),
    "eps=0.1, Q0=0": (run_eps_greedy, dict(epsilon=0.1)),
    "greedy, Q0=15 (optimistic)": (run_eps_greedy, dict(epsilon=0.0, Q0=15.0)),
    "UCB c=2": (run_ucb, dict(c=2.0)),
}
results = {name: average_runs(run, **kw) for name, (run, kw) in configs.items()}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (avg_r, opt) in results.items():
    ax[0].plot(avg_r, label=name); ax[1].plot(opt, label=name)
ax[0].set_xlabel("step"); ax[0].set_ylabel("average reward (100 runs)")
ax[1].set_xlabel("step"); ax[1].set_ylabel("optimal action rate (100 runs)")
ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
for name, (avg_r, opt) in results.items():
    print(f"{name:28s}: 最後の100ステップで最適な台を選んだ割合 {opt[-100:].mean():.3f}")

## 6. 条件を変える

1. $\varepsilon$ を 0.01、0.1、0.3 に変えると、最後の最適な行動の割合はどうなるか。本文 §4 の表の「 $1 - \varepsilon + \varepsilon / K$ 」と比べる
2. 楽観的な初期値を 15 から 5 に下げるとどうなるか(この問題の報酬の平均は 0〜10)
3. UCB の $c$ を 0.5、2、10 に変える

In [ ]:
# 1. epsilon と、最後の最適な行動の割合
for eps in [0.01, 0.1, 0.3]:
    _, opt = average_runs(run_eps_greedy, n_runs=50, epsilon=eps)
    print(f"1. eps = {eps:4}: 最後の100ステップ {opt[-100:].mean():.3f}   (1 - eps + eps/K = {1 - eps + eps / env.K:.3f})")
# 2. 楽観的な初期値
for Q0 in [5.0, 15.0]:
    _, opt = average_runs(run_eps_greedy, n_runs=50, epsilon=0.0, Q0=Q0)
    print(f"2. greedy, Q0 = {Q0:4}: 最後の100ステップ {opt[-100:].mean():.3f}")
# 3. UCB の c
for c in [0.5, 2.0, 10.0]:
    _, opt = average_runs(run_ucb, n_runs=50, c=c)
    print(f"3. UCB c = {c:4}: 最後の100ステップ {opt[-100:].mean():.3f}")

## 7. 報酬の分布が変わる問題:一定のステップサイズ

途中(500 ステップ目)で、いちばん悪い台の平均が、いちばん良い台より大きくなる問題を考える。ステップサイズ $1/N$(標本平均)と、一定の $\alpha = 0.1$ で比べる(本文 §2 の NOTE)。

**予想**:変化のあとで、新しい最適な台に早く気づくのはどちらか。

- 予想:(ここに書く)

In [ ]:
class ChangingBandit(GaussianBandit):
    def reset(self, seed=None):
        super().reset(seed)
        self.t = 0
        self.mu_now = self.mu.copy()
    def step(self, a):
        self.t += 1
        if self.t == 500:                                   # 500ステップ目で入れ替わる
            worst = int(np.argmin(self.mu))
            self.mu_now[worst] = self.mu.max() + 2.0
        return float(self.rng.normal(self.mu_now[a], self.sigma[a]))

cenv = ChangingBandit(seed=2026)
new_best = int(np.argmin(cenv.mu))
for alpha in [None, 0.1]:
    rate = np.mean([run_eps_greedy(cenv, epsilon=0.1, alpha=alpha, seed=k)["actions"][-200:] == new_best for k in range(50)])
    print(f"ステップサイズ {'1/N' if alpha is None else alpha}: 最後の200ステップで新しい最適な台を選んだ割合 {rate:.3f}")

## 8. 研究室のライブラリ `cpai_lab` との関係

`cpai_lab` には、この単元と同じ環境と方法が入っている(本文 §6)。`GaussianBandit` の台の平均と標準偏差は、同じ seed なら、この notebook の環境と同じになるように作ってある。自分の実装と同じ結果になることを確かめる。

In [ ]:
# 研究室のライブラリ cpai_lab を入れる(Colab では、ランタイムをつなぐたびに必要)
%pip install -q git+https://github.com/oit-cpai/lab-core.git

In [ ]:
import cpai_lab
lab_env = cpai_lab.GaussianBandit(seed=2026)
check("台の平均は同じ", lab_env.mu, env.mu)
Q_test = np.array([2., 5., 5., 1., 0.])
check("epsilon_greedy_policy = 自分の epsilon_greedy_probs", cpai_lab.epsilon_greedy_policy(Q_test, 0.1), epsilon_greedy_probs(Q_test, 0.1))

# 同じ seed で1回ずつ実行して、選んだ行動の列が同じになるかを確かめる
lab_env.reset(seed=3)
lab = cpai_lab.value_based_method(lab_env, epsilon=0.1, number_of_steps=300, seed=3, show_plot=False)
mine = run_eps_greedy(env, epsilon=0.1, T=300, seed=3)
check("value_based_method と自分の実装で、選んだ行動が同じ", lab["actions"], mine["actions"])

# run_experiment は、複数回の実行の平均(各時刻の平均の報酬と、最適な行動の割合)を返す
avg_r, opt_rate, _ = cpai_lab.run_experiment(lab_env, cpai_lab.value_based_method, n_runs=50, number_of_steps=1000, epsilon=0.1)
print("cpai_lab.run_experiment(eps=0.1, 50回):最後の100ステップの最適な行動の割合", round(opt_rate[-100:].mean(), 3))

## 9. 説明(自分の言葉で書く)

1. 更新式 $Q \leftarrow Q + \alpha (r - Q)$ は、何をしているか。ステップサイズを $1/N$ にした場合と一定の値にした場合の違いは何か
   - (ここに書く)
2. greedy だけでは良くないのはなぜか。ε-greedy、楽観的な初期値、UCB は、それぞれどうやって探索しているか
   - (ここに書く)
3. 方法を比べるとき、複数回の実行の平均をとるのはなぜか
   - (ここに書く)